# Class likelihood ratios with DummyClassifier baselines: scikit-learn vs Moxie

**Reference example:** `../scikit-learn/examples/model_selection/plot_likelihood_ratios.py` at scikit-learn
`4358542a1bd39af2e7e1cbace8b9fed067473c5b`
([rendered gallery page](https://scikit-learn.org/dev/auto_examples/model_selection/plot_likelihood_ratios.html)).

**Scenario:** a 10%-prevalence binary problem (`make_classification(n_samples=10_000, weights=[0.9, 0.1], random_state=0)`).
The example cross-validates (`cv=10`, i.e. `StratifiedKFold(10)` without shuffling) a custom scorer returning
`class_likelihood_ratios(y, y_pred, replace_undefined_by=1.0)` for
`DummyClassifier(strategy="stratified", random_state=1234)` and `DummyClassifier(strategy="most_frequent")`.

**Adaptations:**
- Data is generated in Python and shared as `.npy`; labels are typed `Label(Int)` values in Moxie.
- Moxie has no `moxie.metrics.class_likelihood_ratios`; the Moxie scorer computes the same confusion-count formula
  of the pinned `sklearn/metrics/_classification.py` (`LR+ = tp*(tn+fp) / (fp*(tp+fn))`,
  `LR- = fn*(tn+fp) / (tn*(tp+fn))`, undefined ratios replaced by 1.0) in Mojo inside a `TypedScorer`.
- Moxie's `cross_validate` takes explicit typed scorers and materialized splits (`StratifiedKFold(10).split(n, y)`),
  per the model-selection family's documented typed interface; the dict-returning Python scorer becomes two scorers.
- The `LogisticRegression` sections, the prevalence sweep, `DecisionBoundaryDisplay` and bootstrap plots are
  run only on the scikit-learn side: Moxie has no `LogisticRegression` (linear-model family), so they are recorded as GAP.

**Determinism:** fully deterministic. The splits are unshuffled, `most_frequent` draws nothing, and the
`stratified` baseline re-seeds `RandomState(1234)` on every `predict`, which Moxie reproduces with the same
MT19937 multinomial stream. Exact parity is therefore required for folds, predictions, priors and ratios.

In [1]:
import sys
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "notebooks/_support/moxie_nb.py").exists())
sys.path.insert(0, str(ROOT / "notebooks/_support"))
%load_ext moxie_nb
nb = moxie_nb.Session("dummy/plot_likelihood_ratios")

## 1. scikit-learn (Python reference)

The example code up to the dummy baselines, unchanged.

In [2]:
import warnings
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification

X, y = make_classification(n_samples=10_000, weights=[0.9, 0.1], random_state=0)
print(f"Percentage of people carrying the disease: {100 * y.mean():.2f}%")

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=0)

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import class_likelihood_ratios

estimator = LogisticRegression().fit(X_train, y_train)
y_pred = estimator.predict(X_test)
pos_LR, neg_LR = class_likelihood_ratios(y_test, y_pred, replace_undefined_by=1.0)
print(f"LR+: {pos_LR:.3f}")


def scoring(estimator, X, y):
    y_pred = estimator.predict(X)
    pos_lr, neg_lr = class_likelihood_ratios(y, y_pred, replace_undefined_by=1.0)
    return {"positive_likelihood_ratio": pos_lr, "negative_likelihood_ratio": neg_lr}


def extract_score(cv_results):
    lr = pd.DataFrame(
        {
            "positive": cv_results["test_positive_likelihood_ratio"],
            "negative": cv_results["test_negative_likelihood_ratio"],
        }
    )
    return lr.aggregate(["mean", "std"])


from sklearn.model_selection import cross_validate

estimator = LogisticRegression()
extract_score(cross_validate(estimator, X, y, scoring=scoring, cv=10))

Percentage of people carrying the disease: 10.37%
LR+: 12.617


,positive,negative
mean,16.661086,0.724702
std,4.383973,0.054045


In [3]:
from sklearn.dummy import DummyClassifier

# return_estimator/return_indices only expose the per-fold state for the parity checks below.
estimator = DummyClassifier(strategy="stratified", random_state=1234)
sk_stratified = cross_validate(estimator, X, y, scoring=scoring, cv=10,
                               return_estimator=True, return_indices=True)
display(extract_score(sk_stratified))

estimator = DummyClassifier(strategy="most_frequent")
with warnings.catch_warnings():
    # No positive predictions: class_likelihood_ratios warns that LR+ is undefined (replaced by 1.0).
    warnings.simplefilter("ignore")
    sk_frequent = cross_validate(estimator, X, y, scoring=scoring, cv=10,
                                 return_estimator=True, return_indices=True)
display(extract_score(sk_frequent))

# Pinned-source cross-check: the installed wheel and the pinned sklearn/dummy.py agree here.
sys.path.insert(0, str(ROOT / "tests/dummy"))
import oracle
pinned = oracle.load()
pinned_scores = cross_validate(pinned.DummyClassifier(strategy="stratified", random_state=1234),
                               X, y, scoring=scoring, cv=10)
assert np.array_equal(pinned_scores["test_positive_likelihood_ratio"],
                      sk_stratified["test_positive_likelihood_ratio"])

fold_of = np.empty(len(y), dtype=np.int64)
for k, test in enumerate(sk_stratified["indices"]["test"]):
    fold_of[test] = k
sk_oof = np.concatenate([est.predict(X[test]) for est, test in
                         zip(sk_stratified["estimator"], sk_stratified["indices"]["test"])])
sk_priors = np.array([est.class_prior_ for est in sk_stratified["estimator"]])
sk_frequent_proba = sk_frequent["estimator"][0].predict_proba(X[sk_frequent["indices"]["test"][0]])
nb.share(X=X, y=y)

,positive,negative
mean,1.108843,0.986989
std,0.268147,0.034278


,positive,negative
mean,1.0,1.0
std,0.0,0.0


## 2. Moxie (native Mojo)

The two DummyClassifier baselines through `moxie.model_selection.cross_validate` with `StratifiedKFold(10)`.

In [4]:
%%mojo likelihood_ratios
from moxie.dummy import DummyClassifier, Label
from moxie.model_selection import TypedScorer, CVAdapter, StratifiedKFold, IndexArray, IndexSplit, cross_validate
from moxie.model_selection._split import _rows
from moxie_nb_io import load, load_ints, save, save_ints
from numojo import NDArray, Shape, Item

comptime Dense = NDArray[DType.float64]
comptime Model = DummyClassifier[]


def as_labels(y: IndexArray) raises -> List[Label]:
    var labels = List[Label]()
    for i in range(y.size):
        labels.append(Label(Int(y[Item(i)])))
    return labels^


struct LikelihoodRatio(TypedScorer, Copyable):
    # class_likelihood_ratios(y, estimator.predict(X), replace_undefined_by=1.0), labels {0, 1}.
    comptime Estimator = Model
    comptime Input = Dense
    comptime Target = IndexArray
    comptime Metadata = Optional[Dense]
    var positive: Bool

    def __init__(out self, positive: Bool):
        self.positive = positive

    def score(self, estimator: Model, X: Dense, y: IndexArray, metadata: Optional[Dense]) raises -> Float64:
        var predicted = estimator.predict(X)
        var tp = 0
        var fp = 0
        var fneg = 0
        var tn = 0
        for i in range(y.size):
            var truth = Int(y[Item(i)]) == 1
            var hit = predicted[i].integer == 1
            if truth and hit:
                tp += 1
            elif truth:
                fneg += 1
            elif hit:
                fp += 1
            else:
                tn += 1
        if tp + fneg == 0:
            raise Error("ValueError: fold has no positive samples")
        if self.positive:
            if fp == 0:
                return 1.0
            return Float64(tp * (tn + fp)) / Float64(fp * (tp + fneg))
        if tn == 0:
            return 1.0
        return Float64(fneg * (tn + fp)) / Float64(tn * (tp + fneg))


struct Adapter(CVAdapter):
    comptime Scorer = LikelihoodRatio

    def __init__(out self):
        pass

    def n_samples(self, X: Dense, y: IndexArray) raises -> Int:
        return y.size

    def input_rows(self, X: Dense, rows: IndexArray, train: IndexArray) raises -> Dense:
        return _rows(X, rows)

    def target_rows(self, y: IndexArray, rows: IndexArray) raises -> IndexArray:
        return _rows(y, rows)

    def metadata_rows(self, metadata: Optional[Dense], rows: IndexArray) raises -> Optional[Dense]:
        if metadata:
            return _rows(metadata.value(), rows)
        return None

    def fit(self, mut estimator: Model, X: Dense, y: IndexArray, metadata: Optional[Dense]) raises:
        _ = estimator.fit(X, as_labels(y))


def main() raises:
    var X = load("X")
    var labels = load_ints("y")
    var n = len(labels)
    var y = IndexArray(Shape(n))
    for i in range(n):
        y[Item(i)] = Int64(labels[i])
    var splits = List[IndexSplit]()
    var cursor = StratifiedKFold(10).split(n, y.copy())
    while True:
        var pair = cursor.next()
        if not pair:
            break
        splits.append(pair.value().copy())
    var fold_of = List[Int]()
    for _ in range(n):
        fold_of.append(-1)
    for k in range(len(splits)):
        for j in range(splits[k].test.size):
            fold_of[Int(splits[k].test[Item(j)])] = k
    save_ints("fold_of", fold_of)

    var scorers = List[LikelihoodRatio]()
    scorers.append(LikelihoodRatio(True))
    scorers.append(LikelihoodRatio(False))
    var names: List[String] = ["positive_likelihood_ratio", "negative_likelihood_ratio"]
    var strategies: List[String] = ["stratified", "most_frequent"]
    for strategy in strategies:
        var seed = Optional[Int](1234) if strategy == "stratified" else Optional[Int]()
        var result = cross_validate[Adapter](
            Model(strategy=strategy, random_state=seed), X, y, Adapter(), scorers, names,
            splits, None, None, return_estimator=True)
        var positive = Dense(Shape(len(splits)))
        var negative = Dense(Shape(len(splits)))
        for k in range(len(splits)):
            positive[Item(k)] = result.scores.test_scores[Item(0, k, 0)]
            negative[Item(k)] = result.scores.test_scores[Item(0, k, 1)]
        save(strategy + "_positive", positive)
        save(strategy + "_negative", negative)
        ref fitted = result.estimators.value()
        if strategy == "stratified":
            var oof = List[Int]()
            var priors = Dense(Shape(len(splits), 2))
            for k in range(len(splits)):
                var predicted = fitted[k].predict(_rows(X, splits[k].test))
                for label in predicted:
                    oof.append(label.integer)
                for c in range(2):
                    priors[Item(k, c)] = fitted[k].class_prior_.item(c)
            save_ints("stratified_oof", oof)
            save("stratified_priors", priors)
        else:
            save("most_frequent_proba", fitted[0].predict_proba(_rows(X, splits[0].test)))
    print("moxie folds:", len(splits))

moxie folds: 10


## 3. Side by side and parity

In [5]:
mx = nb.results["likelihood_ratios"]


def moxie_score(strategy):
    return pd.DataFrame({"positive": mx[f"{strategy}_positive"],
                         "negative": mx[f"{strategy}_negative"]}).aggregate(["mean", "std"])


moxie_nb.side_by_side(sklearn_stratified=extract_score(sk_stratified), moxie_stratified=moxie_score("stratified"))
moxie_nb.side_by_side(sklearn_most_frequent=extract_score(sk_frequent), moxie_most_frequent=moxie_score("most_frequent"))

nb.compare("StratifiedKFold(10) test fold of every row", fold_of, mx["fold_of"], exact=True)
nb.compare("stratified class_prior_ per fold", sk_priors, mx["stratified_priors"], exact=True,
           reason="counts / total in Float64 on both sides")
nb.compare("stratified out-of-fold predictions (seed 1234)", sk_oof, mx["stratified_oof"], exact=True)
for strategy, sk in [("stratified", sk_stratified), ("most_frequent", sk_frequent)]:
    for key in ["positive", "negative"]:
        nb.compare(f"{strategy} test LR{'+' if key == 'positive' else '-'} per fold",
                   sk[f"test_{key}_likelihood_ratio"], mx[f"{strategy}_{key}"], exact=True,
                   reason="integer confusion counts, one Float64 division on both sides")
nb.compare("most_frequent predict_proba (fold 0)", sk_frequent_proba, mx["most_frequent_proba"], exact=True)
nb.note("class_likelihood_ratios", "ADAPTED",
        "no moxie.metrics; the pinned confusion-count formula is evaluated in a Mojo TypedScorer")
nb.note("LogisticRegression sections and prevalence sweep", "GAP",
        "moxie.linear_model has no LogisticRegression (linear-model family); DecisionBoundaryDisplay is not ported")

,positive,negative
mean,1.108843,0.986989
std,0.268147,0.034278
,positive,negative
mean,1.108843,0.986989
std,0.268147,0.034278


,positive,negative
mean,1.0,1.0
std,0.0,0.0
,positive,negative
mean,1.0,1.0
std,0.0,0.0


Moxie cannot run the LogisticRegression sections; the failure is a clear compile-time error:

In [6]:
%%mojo logistic_regression --expect-fail
from moxie.linear_model import LogisticRegression


def main() raises:
    var model = LogisticRegression()

Expected failure (unsupported in Moxie):
compilation failed
/home/strgenix/postdoc/projects/mojods/porting-sklearn/moxie/builds/notebooks/dummy/plot_likelihood_ratios/logistic_regression.mojo:1:32: error: package 'linear_model' does not contain 'LogisticRegression'
from moxie.linear_model import LogisticRegression
                               ^
mojo: error: failed to parse the provided Mojo source module



In [7]:
nb.report()
nb.assert_parity()

check,status,sklearn shape,moxie shape,max |diff|,max rel,tolerance,detail,tolerance reason
StratifiedKFold(10) test fold of every row,PASS,"(10000,)","(10000,)",,,exact,0 mismatches,
stratified class_prior_ per fold,PASS,"(10, 2)","(10, 2)",,,exact,0 mismatches,counts / total in Float64 on both sides
stratified out-of-fold predictions (seed 1234),PASS,"(10000,)","(10000,)",,,exact,0 mismatches,
stratified test LR+ per fold,PASS,"(10,)","(10,)",,,exact,0 mismatches,"integer confusion counts, one Float64 division on both sides"
stratified test LR- per fold,PASS,"(10,)","(10,)",,,exact,0 mismatches,"integer confusion counts, one Float64 division on both sides"
most_frequent test LR+ per fold,PASS,"(10,)","(10,)",,,exact,0 mismatches,"integer confusion counts, one Float64 division on both sides"
most_frequent test LR- per fold,PASS,"(10,)","(10,)",,,exact,0 mismatches,"integer confusion counts, one Float64 division on both sides"
most_frequent predict_proba (fold 0),PASS,"(1000, 2)","(1000, 2)",,,exact,0 mismatches,
class_likelihood_ratios,ADAPTED,,,,,,no moxie.metrics; the pinned confusion-count formula is evaluated in a Mojo TypedScorer,
LogisticRegression sections and prevalence sweep,GAP,,,,,,moxie.linear_model has no LogisticRegression (linear-model family); DecisionBoundaryDisplay is not ported,
